# 1. Load and inspect the travel bookings
# Replace the catalog and schema with your actual names.

In [0]:
table_name = "/Volumes/damo630_fall_2026/default/multicity_project/multi_city_booking.csv"

bookings_df = spark.read.csv(table_name, header=True, inferSchema=True)

print("Total booking records:", bookings_df.count())

bookings_df.printSchema()

display(bookings_df.limit(10))


## 2. Data Quality Assessment

Before conducting affinity analysis, the dataset
must be evaluated for completeness and consistency.

This stage examines:
- Missing values across all columns
- Number of unique trips and destination cities
- Booking date coverage

These checks help determine whether the data
is suitable for transaction-based analysis.

In [0]:

from pyspark.sql import functions as F

# 2.1 Check missing values across all columns
missing_values = bookings_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in bookings_df.columns
])

print("Missing values by column:")
display(missing_values)

# 2.2 Summarize trips, cities, and booking dates
dataset_summary = bookings_df.agg(
    F.countDistinct("utrip_id").alias("unique_trips"),
    F.countDistinct("city_id").alias("unique_cities"),
    F.min("checkin").alias("earliest_checkin"),
    F.max("checkout").alias("latest_checkout")
)

print("Dataset summary:")
display(dataset_summary)



## 3. Transaction Preparation

The booking-level dataset is transformed into
trip-level transactions for affinity analysis.

Each unique trip ID represents one transaction.
Its basket contains the distinct destination cities
booked within that trip.

Duplicate city visits within the same trip are
counted only once.

In [0]:

from pyspark.sql import functions as F

# 3.1 Check records with invalid trip/city identifiers
quality_checks = bookings_df.agg(
    F.sum(
        (F.col("utrip_id").isNull() |
         (F.trim(F.col("utrip_id")) == "")).cast("int")
    ).alias("invalid_trip_ids"),
    F.sum(
        F.col("city_id").isNull().cast("int")
    ).alias("missing_city_ids"),
    F.sum(
        (F.col("checkout") < F.col("checkin")).cast("int")
    ).alias("invalid_date_order")
)

display(quality_checks)

# 3.2 Keep records with valid transaction identifiers
valid_bookings_df = bookings_df.filter(
    F.col("utrip_id").isNotNull() &
    (F.trim(F.col("utrip_id")) != "") &
    F.col("city_id").isNotNull()
)

# 3.3 Build distinct city baskets
transactions_df = (
    valid_bookings_df
    .groupBy("utrip_id")
    .agg(
        F.array_sort(
            F.collect_set("city_id")
        ).alias("cities")
    )
    .withColumn("basket_size", F.size("cities"))
)

# 3.4 Inspect example baskets without displaying user IDs
display(
    transactions_df
    .select("cities", "basket_size")
    .limit(10)
)

# 3.5 Summarize transaction sizes
basket_summary = transactions_df.agg(
    F.count("*").alias("total_transactions"),
    F.round(F.avg("basket_size"), 2).alias("avg_cities_per_trip"),
    F.min("basket_size").alias("min_cities"),
    F.max("basket_size").alias("max_cities")
)

display(basket_summary)


## 4. Exploratory Data Analysis

### 4.1 Distribution of Cities per Trip

This analysis examines the number of unique destination
cities included in each trip.

The goal is to understand travel patterns and determine
whether the dataset is suitable for association rule mining.

Trips containing only one city cannot produce
city-to-city association rules.

In [0]:

from pyspark.sql import functions as F
import matplotlib.pyplot as plt

# 4.1 Summary statistics
trip_stats = transactions_df.agg(
    F.count("*").alias("total_trips"),
    F.round(F.avg("basket_size"), 2).alias("avg_cities_per_trip"),
    F.max("basket_size").alias("max_cities_per_trip"),
    F.sum(F.when(F.col("basket_size") == 1, 1)
          .otherwise(0)).alias("single_city_trips"),
    F.sum(F.when(F.col("basket_size") >= 2, 1)
          .otherwise(0)).alias("multi_city_trips")
)

display(trip_stats)

# 4.2 Group trip sizes for visualization
# Trips with 11 or more cities are grouped together.

basket_distribution = (
    transactions_df
    .withColumn(
        "size_group",
        F.when(F.col("basket_size") >= 11, 11)
         .otherwise(F.col("basket_size"))
    )
    .groupBy("size_group")
    .count()
    .orderBy("size_group")
)

# Only a small aggregated table is collected to Python
plot_data = basket_distribution.collect()

labels = [
    "11+" if row["size_group"] == 11
    else str(row["size_group"])
    for row in plot_data
]

counts = [row["count"] for row in plot_data]

# 4.3 Visualization
plt.figure(figsize=(10, 5))
plt.bar(labels, counts)

plt.title("Distribution of Unique Cities per Trip")
plt.xlabel("Number of Unique Cities")
plt.ylabel("Number of Trips")

plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()